In [ ]:
# Cell 1 — clone/pull + install
!git clone https://github.com/Jaswanth-K1210/SDAM.git 2>/dev/null || (cd SDAM && git pull --no-edit origin main)
%cd SDAM
!pip install -q -r probe/requirements.txt
print("Setup complete")

In [ ]:
# Cell 2 — GPU check (expect A100)
import torch
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Cell 3 — CLEVR v1.0 (images + scene graphs), ~18GB, once
import os
if not os.path.exists("data/CLEVR_v1.0/scenes/CLEVR_train_scenes.json"):
    os.makedirs("data", exist_ok=True)
    !wget -q --show-progress https://dl.fbaipublicfiles.com/clevr/CLEVR_v1.0.zip -O data/CLEVR_v1.0.zip
    !cd data && unzip -q CLEVR_v1.0.zip && rm -f CLEVR_v1.0.zip
print("scenes:", os.path.exists("data/CLEVR_v1.0/scenes/CLEVR_train_scenes.json"),
      "| images:", os.path.isdir("data/CLEVR_v1.0/images/train"))

In [ ]:
# Cell 4 — test suites must be GREEN before the pipeline runs
!python -m pytest tests/test_variance.py tests/test_decodability.py tests/test_clevr_factors.py tests/test_cosine_check.py tests/test_variance_gate.py tests/test_run_probe.py tests/test_feature_cache.py tests/test_objectness_pipeline.py -q

In [ ]:
# Cell 5 — run the objectness pipeline (Phase 1 adjudicates; Phase 2 diagnostic)
# First run extracts + caches DINOv2 features (~minutes on A100); reused after.
!python -m experiments.objectness_pipeline

In [ ]:
# Cell 6 — Phase 1 plot
from IPython.display import Image, display
display(Image("results/objectness_phase1.png"))

In [ ]:
# Cell 7 — full results JSON
print(open("results/objectness_pipeline.json").read())

In [ ]:
# Cell 8 — commit results to GitHub
from google.colab import userdata
import os
try:
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = ""
!git config --global user.email "jaswanthkoppisetty@gmail.com"
!git config --global user.name "Jaswanth-K1210"
!mkdir -p results_archive/objectness
!cp -f results/objectness_pipeline.json results/objectness_phase1.png results_archive/objectness/ 2>/dev/null || true
!git add results_archive/objectness/
!git commit -m "results(objectness): Phase 1 retrieval + Phase 2 diagnostic run" || echo "nothing to commit"
if token:
    os.system(f"git remote set-url origin https://{token}@github.com/Jaswanth-K1210/SDAM.git")
    os.system("git push origin main")
    print("Pushed.")
else:
    print("Add GITHUB_TOKEN to Colab Secrets then re-run.")